# IDB Early Alert Screening Tool (EAST) Dashboard Prototype

Steps:
1) Set up hosted layer for earthquake events 
    - Connect to GDACS API
    - Filter LAC Countries and recent events
    - Set up thresholds
    - Set up backup source
2) Repeat for droughts and floods
3) Set up email alert 

In [ ]:
COUNTRIES = {"Argentina","Bahamas","Barbados","Belize","Bolivia","Brazil","Chile",
             "Colombia", "Costa Rica","Dominican Republic","Ecuador","El Salvador",
             "Guatemala","Guyana","Haiti","Honduras","Jamaica","Mexico",
             "Nicaragua","Panama","Paraguay","Peru","Suriname",
             "Trinidad and Tobago","Uruguay","Venezuela"}

ISO3 = {"ARG","BHS","BRB","BLZ","BOL","BRA","CHL","COL","CRI","DOM","ECU","SLV",
        "GTM","GUY","HTI","HND","JAM","MEX","NIC","PAN","PRY","PER","SUR","TTO",
        "URY","VEN"}

LAC_BOUNDS = {
    "minlatitude": -56.0,  # Southern tip of Chile/Argentina
    "maxlatitude": 33.0,  # Northern Mexico
    "minlongitude": -118.0,  # Western Mexico
    "maxlongitude": -34.0,  # Eastern Brazil
}

DAYS_TO_PULL = 30

# Hazards

### Earthquakes

In [2]:
# Imports
from datetime import datetime, timedelta
import requests

In [3]:
today = datetime.now()
one_month_ago = today - timedelta(days=30)

from_date = one_month_ago.strftime("%Y-%m-%d")
to_date = today.strftime("%Y-%m-%d")

countries_param = ",".join(COUNTRIES)

In [ ]:
def classify_earthquake_alert(magnitude):
    """
    Classifies earthquake threats based strictly on Richter/Moment Magnitude (Mw).
    - Green:  < 5.0  (Minor / Light)
    - Orange: 5.0 - 6.4 (Moderate / Strong)
    - Red:    >= 6.5 (Major / Great)
    """
    if magnitude is None:
        return "Green"
    
    try:
        mag = float(magnitude)
        if mag >= 6.5:
            return "Red"
        elif mag >= 5.0:
            return "Orange"
        else:
            return "Green"
    except (ValueError, TypeError):
        return "Green"

In [ ]:
def fetch_earthquake_data():
    """
    Write note
    """
    
    BASE_URL = "https://www.gdacs.org/gdacsapi/api/Events/geteventlist/search"

    params = {
        "eventlist": "EQ",
        "fromdate": from_date,
        "todate": to_date, 
        "country": countries_param
    }

    print(f"Attempting to pull Primary Source (GDACS) from {from_date} to {to_date}...")
    try:
        response = requests.get(BASE_URL, params=params, timeout=15)
        response.raise_for_status()
        data = response.json()
        events = data.get("features", data)
        return events, "GDACS"
    
    except Exception as primary_error:
        print(f"!! Primary Source (GDACS) Failed: {primary_error}")
        print("Executing Fallback Logic (EMSC)...")
        
        # --- PHASE 2 FALLBACK PLACEHOLDER ---
        # TODO: Implement your EMSC API request here when ready
        # raw_emsc_data = requests.get(EMSC_URL)...
        # return emsc_events, "EMSC"
        return [], "NONE"

events, source_used = fetch_earthquake_data()

In [ ]:
# Backup Source for Earthquake
def earthquake_emsc_backup():
    
    EMSC_URL = "https://www.seismicportal.eu/fdsnws/event/1/query"

    params = {
        "format": "json",
        "starttime": f"{start_date_str}T00:00:00",
        "endtime": f"{end_date_str}T23:59:59",
        "minmagnitude": "3.5",  # Filter out minor micro-tremors
        **LAC_BOUNDS,
    }

    print("Executing Fallback: Querying EMSC (SeismicPortal) API...")

    try:
        response = requests.get(EMSC_URL, params=params, timeout=15)
        response.raise_for_status()
        data = response.json()

        events = data.get("features", [])
        mapped_records = []

        for event in events:
            props = event.get("properties", {})
            geometry = event.get("geometry", {})
            coords = geometry.get("coordinates", [None, None, None])

            # EMSC uses 'unid' or 'evid' as unique identifiers
            emsc_id = props.get("unid") or props.get("evid") or event.get("id")
            unique_event_id = f"EMSC-EQ-{emsc_id}"

            # Parse Timestamps (EMSC uses ISO 8601: "2026-06-24T22:05:12.4Z")
            raw_time = props.get("time")
            incident_date, incident_time = None, None
            if raw_time:
                try:
                    dt_obj = datetime.strptime(
                        raw_time.split(".")[0].replace("Z", ""),
                        "%Y-%m-%dT%H:%M:%S",
                    )
                    incident_date = dt_obj.strftime("%Y-%m-%d")
                    incident_time = dt_obj.strftime("%H:%M:%S")
                except ValueError:
                    incident_date = raw_time

            # Compute Alert Level derived from magnitude
            magnitude = props.get("mag")
            alert_level = classify_earthquake_alert(magnitude)

            # Extract location string (e.g., "OFF COAST OF VENEZUELA")
            flynn_region = props.get("flynn_region", "")

            # EMSC GeoJSON coordinates array: [longitude, latitude, depth_km]
            depth_km = coords[2] if len(coords) > 2 else props.get("depth")

            master_row = {
                "event_id": unique_event_id,
                "episode_id": unique_event_id,
                "hazard_type": "Earthquake",
                "alert_level": alert_level,
                "raw_intensity": magnitude,
                "intensity_unit": f"Magnitude ({props.get('magtype', 'M')})",
                "latitude": coords[1],
                "longitude": coords[0],
                "country_primary_name": flynn_region,  # Region description
                "country_primary_iso": None,  # EMSC FDSN raw feed omits ISO3
                "countries_all": None,
                "incident_date": incident_date,
                "incident_time": incident_time,
                "update_date": incident_date,  # Uses origin date if last modified is unlisted
                "update_time": incident_time,
                "source_primary": "GDACS",
                "source_used": "EMSC",  # Explicitly flags that fallback was triggered
                "source_report_url": f"https://www.emsc-csem.org/Earthquake/earthquake.php?id={emsc_id}",
                "alerted": 0,
                # Hazard-Specific Fields
                "magnitude": magnitude,
                "depth_km": depth_km,
                "tsunami_alert": 0,
                # Unused for Earthquakes
                "category": None,
                "wind_speed": None,
                "spi_value": None,
                "discharge_return_period_yrs": None,
            }

            mapped_records.append(master_row)

        print(
            f"Successfully processed {len(mapped_records)} records from EMSC."
        )
        return mapped_records

    except Exception as e:
        print(f"!! Fallback Source (EMSC) also failed: {e}")
        return []

In [ ]:
import re

# Simulating the raw features list from JSON payload
mapped_records = []

for event in events:
    props = event.get("properties", {})
    geometry = event.get("geometry", {})
    coords = geometry.get("coordinates", [None, None])
    
    # Nested URL extraction
    url_block = props.get("url", {})
    report_url = url_block.get("report")
    
    # ID Generation based on your schema rule
    gdacs_id = props.get("eventid")
    gdacs_episode = props.get("episodeid")
    unique_event_id = f"GDACS-EQ-{gdacs_id}"
    unique_episode_id = f"GDACS-EQ-{gdacs_id}-{gdacs_episode}" if gdacs_episode else unique_event_id
    
    # Process the multi-country affected list into a comma-separated string of ISO3s
    affected_list = props.get("affectedcountries", [])
    all_iso3_codes = [c.get("iso3") for c in affected_list if c.get("iso3")]
    countries_all_string = ", ".join(all_iso3_codes)
    
    # Parse Timestamps (Splitting Date and Time)
    def split_datetime(dt_str):
        if not dt_str:
            return None, None
        try:
            # Handle standard ISO 'T' separator
            dt_obj = datetime.strptime(dt_str.split(".")[0].replace("Z", ""), "%Y-%m-%dT%H:%M:%S")
            return dt_obj.strftime("%Y-%m-%d"), dt_obj.strftime("%H:%M:%S")
        except ValueError:
            return dt_str, None

    incident_date, incident_time = split_datetime(props.get("fromdate"))
    update_date, update_time = split_datetime(props.get("datemodified"))
    
    # Intensity & Severity Details
    severity_data = props.get("severitydata", {}) or {}
    magnitude = severity_data.get("severity")
    intensity_unit = severity_data.get("severityunit")
    alert_level = classify_earthquake_alert(magnitude)
    
    # Extract depth from text string (e.g., "Magnitude 7.5M, Depth:10km")
    severity_text = severity_data.get("severitytext", "")
    depth_km = None
    if "Depth:" in severity_text:
        try:
            # Look for numbers between 'Depth:' and 'km'
            match = re.search(r"Depth:\s*([\d.]+)\s*km", severity_text)
            if match:
                depth_km = float(match.group(1))
        except Exception:
            pass # Keep as None if string layout shifts anomalously
            
    # Check for Tsunami mentions in descriptions
    html_desc = props.get("htmldescription", "").lower()
    tsunami_flag = 1 if "tsunami" in html_desc else 0

    # Strict Schema row compilation matching your layout
    master_row = {
        "event_id": unique_event_id,
        "episode_id": unique_episode_id,
        "hazard_type": "Earthquake",
        "alert_level": alert_level
        "raw_intensity": magnitude,
        "intensity_unit": f"Magnitude ({intensity_unit})" if intensity_unit else "Magnitude",
        "latitude": coords[1],  # GeoJSON index 1 is Lat
        "longitude": coords[0], # GeoJSON index 0 is Lon
        "country_primary_name": props.get("country"),
        "country_primary_iso": props.get("iso3"),
        "countries_all": countries_all_string,
        "incident_date": incident_date,
        "incident_time": incident_time,
        "update_date": update_date,
        "update_time": update_time,
        "source_primary": "GDACS",
        "source_used": "GDACS", # This will dynamicize when fallback layers are wired
        "source_report_url": report_url,
        "alerted": 0,
        # Hazard-Specific Fields
        "magnitude": magnitude,
        "depth_km": depth_km,
        "tsunami_alert": tsunami_flag,
        # Unused for this hazard archetype
        "category": None,
        "wind_speed": None,
        "spi_value": None,
        "discharge_return_period_yrs": None
    }
    
    mapped_records.append(master_row)

print(f"Successfully processed and mapped {len(mapped_records)} rows into Master Schema format.")

### Floods

### Droughts

## ArcGIS Hosted Layer

In [3]:
from arcgis.gis import GIS
from arcgis.features import FeatureLayer

In [ ]:
# --- 1. Connect to ArcGIS Online ---
# For Phase 2 manual testing, you can log in interactively. 
# For Phase 4 automation, you'll shift to using a client_id/client_secret or a saved profile.
print("Connecting to ArcGIS Online...")
gis = GIS("https://www.arcgis.com", username="YOUR_ARCGIS_USERNAME", password="YOUR_ARCGIS_PASSWORD")

# --- 2. Target your Hosted Feature Layer ---
# You can find this Item ID in the URL of your hosted feature layer's details page on AGOL.
LAYER_ITEM_ID = "f8be4692adeb49fda106ebaa602b360d"
layer_item = gis.content.get(LAYER_ITEM_ID)

# Grab the first layer inside the feature service (index 0)
feature_layer = layer_item.layers[0]
print(f"Connected to layer: {layer_item.title}")

# --- 3. Query Existing Event IDs to Detect Duplicates ---
# We pull down all event_ids currently sitting in the feature layer to build a lookup list.
print("Querying existing event IDs for deduplication...")
existing_features = feature_layer.query(
    where="1=1", 
    out_fields=["event_id", "OBJECTID"] # Only grab what we need to minimize network load
)

# Map existing event_ids to their internal ArcGIS OBJECTID for easy updating
# Structure: {"GDACS-EQ-12345": 99}
existing_id_map = {
    f.attributes["event_id"]: f.attributes["OBJECTID"] 
    for f in existing_features.features 
    if "event_id" in f.attributes
}

# --- 4. Sort Mapped Records into 'Adds' vs 'Updates' ---
# Assume `mapped_records` is the list of dictionaries generated from your GDACS step
features_to_add = []
features_to_update = []

for record in mapped_records:
    event_id = record["event_id"]
    
    # Structure the payload exactly how the ArcGIS REST API expects a feature
    feature_payload = {
        "geometry": {
            "x": record["longitude"],
            "y": record["latitude"],
            "spatialReference": {"wkid": 4326} # Standard WGS84 Geographic Coordinates
        },
        "attributes": record # Your flat schema dictionary matches your layer's field names
    }
    
    # DEDUPLICATION LOGIC
    if event_id in existing_id_map:
        # If it exists, inject the existing OBJECTID so ArcGIS knows which row to overwrite
        feature_payload["attributes"]["OBJECTID"] = existing_id_map[event_id]
        features_to_update.append(feature_payload)
    else:
        features_to_add.append(feature_payload)

print(f"Categorized records: {len(features_to_add)} to ADD, {len(features_to_update)} to UPDATE.")

# --- 5. Commit the Changes via Edit Features ---
if features_to_add or features_to_update:
    try:
        print("Pushing data to ArcGIS Online hosted layer...")
        result = feature_layer.edit_features(
            adds=features_to_add,
            updates=features_to_update,
            rollback_on_failure=True # Ensures transactional safety
        )
        
        # --- 6. Verify Results ---
        add_results = result.get("addResults", [])
        update_results = result.get("updateResults", [])
        
        successful_adds = sum(1 for r in add_results if r.get("success"))
        successful_updates = sum(1 for r in update_results if r.get("success"))
        
        print(f"Successfully ADDED {successful_adds}/{len(features_to_add)} new events.")
        print(f"Successfully UPDATED {successful_updates}/{len(features_to_update)} existing events.")
        
        # Log any specific row errors returned by Esri
        for r in add_results + update_results:
            if not r.get("success"):
                print(f"Row Error: {r.get('error')}")
                
    except Exception as e:
        print(f"An error occurred while communicating with ArcGIS REST API: {e}")
else:
    print("No new or modified events found. Layer is up to date.")